# Pipe-1: Complete End-to-End ML Pipeline
## Synthetic Image Attribution Challenge

**Complete workflow:** Data Loading → EDA → Preprocessing → CV Split → Model Training → Validation → Checkpoint Selection → Inference & Submission

Follow the README.md alongside this notebook for detailed explanations.

## Setup & Configuration

In [ ]:
# Install required packages (if needed)
import subprocess
import sys

packages = [
    'torch',
    'torchvision',
    'timm',
    'albumentations',
    'pandas',
    'numpy',
    'pillow',
    'matplotlib',
    'seaborn',
    'scikit-learn'
]

print("Checking/installing packages...")
for package in packages:
    try:
        __import__(package.replace('-', '_'))
        print(f"✓ {package}")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
        print(f"✓ {package} installed")

print("\n✓ All packages available")

In [ ]:
# EXACT: Import all required libraries
import os
import sys
import json
import csv
import time
import logging
import shutil
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
import timm

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

print("✓ All imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# EXACT: Setup logging and working directory (Kaggle environment)
os.chdir('/kaggle/working')

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler('/kaggle/working/pipeline.log')
    ]
)
logger = logging.getLogger(__name__)

logger.info(f"Working directory: {os.getcwd()}")
logger.info(f"Notebook: Pipe-1 Complete End-to-End Pipeline")
logger.info(f"Timestamp: {datetime.now()}")

In [ ]:
# EXACT: Configuration (hardcoded, don't change)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f"Device: {DEVICE}")

# Preprocessing
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
TARGET_SIZE = 224

# Models
MODELS = {

    # =========================================================
    # Ultra-Scale Leaders (Highest Capacity)
    # =========================================================

    1: 'tf_efficientnet_l2.ns_jft_in1k',
    2: 'tf_efficientnetv2_xl.in21k_ft_in1k',
    3: 'tf_efficientnetv2_l.in21k_ft_in1k',

    # =========================================================
    # High-Tier Heavyweights
    # =========================================================

    4: 'tf_efficientnet_b7.ns_jft_in1k',
    5: 'tf_efficientnet_b7.ra_in1k',
    6: 'tf_efficientnetv2_m.in21k_ft_in1k',

    # =========================================================
    # Mid-Tier Production Standards
    # =========================================================

    7: 'tf_efficientnet_b6.ns_jft_in1k',
    8: 'tf_efficientnet_b5.ns_jft_in1k',
    9: 'tf_efficientnetv2_s.in21k_ft_in1k',
    10: 'tf_efficientnet_b4.ns_jft_in1k',

    # =========================================================
    # Low-Mid Tier (Balanced Compute)
    # =========================================================

    11: 'tf_efficientnet_b3.ns_jft_in1k',
    12: 'tf_efficientnet_b2.ns_jft_in1k',
    13: 'tf_efficientnet_b1.ns_jft_in1k',

    # =========================================================
    # Entry Tier / Lightweight
    # =========================================================

    14: 'tf_efficientnet_b0.ns_jft_in1k',
    15: 'tf_efficientnetv2_b0.in1k',
    16: 'tf_efficientnet_lite4.in1k',
    17: 'tf_efficientnet_lite0.in1k',

}

MODEL_NAME = MODELS[6]
PRETRAINED = True
NUM_CLASSES = 10

# Training
EPOCHS = 25
BATCH_SIZE_TRAIN = 32
BATCH_SIZE_VAL = 32
LEARNING_RATE = 1e-4
DROPOUT_RATE = 0.45
DROPOUT_PATH_RATE = 0.25
WEIGHT_DECAY = 1e-5
LABEL_SMOOTHING = 0.1
BATA = (0.9, 0.999)
EPS = 1e-8
T_MAX = EPOCHS

logger.info(f"""
Configuration:
  Device: {DEVICE}
  Model: {MODEL_NAME} (pretrained={PRETRAINED})
  Epochs: {EPOCHS}
  Batch size: train={BATCH_SIZE_TRAIN}, val={BATCH_SIZE_VAL}
  Learning rate: {LEARNING_RATE}
  Target size: {TARGET_SIZE}x{TARGET_SIZE}
""")

---
# STAGE 1: Data Loading & Validation

In [ ]:
# STAGE 1: EXACT - Load CSVs (Kaggle structure)
logger.info("\n" + "="*60)
logger.info("STAGE 1: DATA LOADING & VALIDATION")
logger.info("="*60)

BASE_INPUT = '/kaggle/input/competitions/dlmmdd-workshop-synthetic-source-attribution-challenge/Data/'

train_df = pd.read_csv(f'{BASE_INPUT}Data/training.csv')
test_df = pd.read_csv(f'{BASE_INPUT}Data/test.csv')

logger.info(f"Loaded training.csv: {train_df.shape}")
logger.info(f"Loaded test.csv: {test_df.shape}")
logger.info(f"Train columns: {list(train_df.columns)}")
logger.info(f"Test columns: {list(test_df.columns)}")

# Verify column names
assert list(train_df.columns) == ['ID', 'path', 'y'], f"Got columns: {list(train_df.columns)}"
assert list(test_df.columns) == ['ID', 'path'], f"Got columns: {list(test_df.columns)}"

logger.info("✓ Column names verified")

In [ ]:
# STAGE 1: EXACT - Verify file existence
logger.info("\nVerifying file existence...")

train_missing = []
for idx, row in train_df.iterrows():
    path = f"{BASE_INPUT}{row['path']}"
    if not os.path.exists(path):
        train_missing.append(path)

if train_missing:
    raise FileNotFoundError(f"Missing {len(train_missing)} training files: {train_missing[:5]}")
else:
    logger.info(f"✓ All {len(train_df)} training files exist")

test_missing = []
for idx, row in test_df.iterrows():
    path = f"{BASE_INPUT}{row['path']}"
    if not os.path.exists(path):
        test_missing.append(path)

if test_missing:
    raise FileNotFoundError(f"Missing {len(test_missing)} test files: {test_missing[:5]}")
else:
    logger.info(f"✓ All {len(test_df)} test files exist")

In [ ]:
# STAGE 1: EXACT - Extract image metadata
logger.info("\nExtracting image metadata...")

# Create class index to name mapping
class_idx_to_name = {
    0: 'AuraFlow',
    1: 'Freepik',
    2: 'Lumina',
    3: 'Photon',
    4: 'Pixart-sigma',
    5: 'Playground v2.5',
    6: 'StableDiffusion3',
    7: 'StableDiffusion3.5',
    8: 'StableDiffusionXL-Turbo',
    9: 'Tencent Hunyuan'
}

train_metadata = []
for idx, row in train_df.iterrows():
    path = f"{BASE_INPUT}{row['path']}"
    img = Image.open(path)
    size_bytes = os.path.getsize(path)
    train_metadata.append({
        'image_id': row['ID'],
        'source': class_idx_to_name[row['y']],
        'width': img.width,
        'height': img.height,
        'format': img.format,
        'size_kb': size_bytes / 1024,
        'mode': img.mode
    })
    if (idx + 1) % 1000 == 0:
        logger.info(f"Processed {idx + 1} training images")

train_metadata_df = pd.DataFrame(train_metadata)
logger.info(f"✓ Metadata extracted for {len(train_metadata_df)} images")

In [ ]:
# STAGE 1: EXACT - Validate class distribution
logger.info("\nValidating class distribution...")

class_counts = train_metadata_df['source'].value_counts()
logger.info(f"\nClass distribution:\n{class_counts}")

# Verify each class has exactly 700 (7000 total / 10 classes)
expected_per_class = 700
for class_name, count in class_counts.items():
    assert count == expected_per_class, f"Class {class_name} has {count} images, expected {expected_per_class}"

logger.info(f"✓ All {len(class_counts)} classes have exactly {expected_per_class} images ({len(class_counts) * expected_per_class} total)")

# Verify no duplicate IDs
assert len(train_df) == len(train_df['ID'].unique()), "Duplicate image IDs found"
logger.info("✓ No duplicate image IDs")

---
# STAGE 2: Exploratory Data Analysis (EDA)

In [ ]:
# STAGE 2: EXACT - Setup EDA outputs
logger.info("\n" + "="*60)
logger.info("STAGE 2: EXPLORATORY DATA ANALYSIS (EDA)")
logger.info("="*60)

os.makedirs('/kaggle/working/eda_outputs', exist_ok=True)
os.makedirs('/kaggle/working/eda_outputs/plots', exist_ok=True)

logger.info("Created eda_outputs directory")

In [ ]:
# STAGE 2: EXACT - Class distribution analysis
logger.info("\nClass distribution analysis...")

fig, ax = plt.subplots(figsize=(12, 6))
class_counts.plot(kind='bar', ax=ax, color='steelblue')
ax.set_xlabel('Generator Class')
ax.set_ylabel('Number of Images')
ax.set_title('Training Data: Class Distribution')
ax.set_ylim([900, 1100])
for i, v in enumerate(class_counts):
    ax.text(i, v + 10, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/eda_outputs/class_distribution.png', dpi=100, bbox_inches='tight')
plt.close()

logger.info("✓ Saved class_distribution.png")

In [ ]:
# STAGE 2: EXACT - Image dimension analysis
logger.info("Image dimension analysis...")

logger.info(f"Width:  mean={train_metadata_df['width'].mean():.1f}, std={train_metadata_df['width'].std():.1f}")
logger.info(f"        min={train_metadata_df['width'].min()}, max={train_metadata_df['width'].max()}")
logger.info(f"Height: mean={train_metadata_df['height'].mean():.1f}, std={train_metadata_df['height'].std():.1f}")
logger.info(f"        min={train_metadata_df['height'].min()}, max={train_metadata_df['height'].max()}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(train_metadata_df['width'], bins=50, color='skyblue', edgecolor='black')
axes[0].set_xlabel('Width (pixels)')
axes[0].set_ylabel('Count')
axes[0].set_title('Width Distribution')
axes[1].hist(train_metadata_df['height'], bins=50, color='lightcoral', edgecolor='black')
axes[1].set_xlabel('Height (pixels)')
axes[1].set_ylabel('Count')
axes[1].set_title('Height Distribution')
plt.tight_layout()
plt.savefig('/kaggle/working/eda_outputs/plots/dimension_distribution.png', dpi=100, bbox_inches='tight')
plt.close()

logger.info("✓ Saved dimension_distribution.png")

In [ ]:
# STAGE 2: EXACT - File format and size analysis
logger.info("File format and size analysis...")

format_counts = train_metadata_df['format'].value_counts()
logger.info(f"Image formats:\n{format_counts}")

logger.info(f"Image size (KB) - Mean: {train_metadata_df['size_kb'].mean():.2f}")
logger.info(f"                  Std:  {train_metadata_df['size_kb'].std():.2f}")
logger.info(f"                  Min:  {train_metadata_df['size_kb'].min():.2f}")
logger.info(f"                  Max:  {train_metadata_df['size_kb'].max():.2f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot([train_metadata_df[train_metadata_df['source'] == src]['size_kb'].values 
            for src in sorted(train_metadata_df['source'].unique())],
           labels=sorted(train_metadata_df['source'].unique()))
ax.set_ylabel('File Size (KB)')
ax.set_title('File Size Distribution by Generator')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('/kaggle/working/eda_outputs/plots/filesize_boxplot.png', dpi=100, bbox_inches='tight')
plt.close()

logger.info("✓ Saved filesize_boxplot.png")

In [ ]:
# STAGE 2: EXACT - Generator sample grid
logger.info("Generator sample grid visualization...")

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()

classes = sorted(train_metadata_df['source'].unique())
for col_idx, class_name in enumerate(classes):
    class_images = train_metadata_df[train_metadata_df['source'] == class_name]
    sample_row = class_images.sample(1).iloc[0]
    
    # Find matching row in train_df to get path
    train_row = train_df[train_df['ID'] == sample_row['image_id']].iloc[0]
    img_path = f"{BASE_INPUT}{train_row['path']}"
    img = Image.open(img_path)
    
    axes[col_idx].imshow(img)
    axes[col_idx].set_title(class_name, fontsize=10, fontweight='bold')
    axes[col_idx].axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/eda_outputs/plots/generator_samples.png', dpi=100, bbox_inches='tight')
plt.close()

logger.info("✓ Saved generator_samples.png")

In [ ]:
# STAGE 2: EXACT - Save EDA insights
logger.info("Saving EDA insights...")

eda_insights = {
    'class_distribution': class_counts.to_dict(),
    'image_dimensions': {
        'width': {
            'mean': float(train_metadata_df['width'].mean()),
            'std': float(train_metadata_df['width'].std()),
            'min': int(train_metadata_df['width'].min()),
            'max': int(train_metadata_df['width'].max())
        },
        'height': {
            'mean': float(train_metadata_df['height'].mean()),
            'std': float(train_metadata_df['height'].std()),
            'min': int(train_metadata_df['height'].min()),
            'max': int(train_metadata_df['height'].max())
        }
    },
    'file_format': format_counts.to_dict(),
    'file_size_kb': {
        'mean': float(train_metadata_df['size_kb'].mean()),
        'std': float(train_metadata_df['size_kb'].std()),
        'min': float(train_metadata_df['size_kb'].min()),
        'max': float(train_metadata_df['size_kb'].max())
    },
    'image_modes': train_metadata_df['mode'].value_counts().to_dict(),
    'num_training_images': len(train_df),
    'num_test_images': len(test_df),
    'num_classes': len(class_counts)
}

with open('/kaggle/working/eda_outputs/eda_insights.json', 'w') as f:
    json.dump(eda_insights, f, indent=2)

logger.info("✓ Saved eda_insights.json")
logger.info("\n✓ STAGE 2 COMPLETE: EDA finished")

---
# STAGE 3: Preprocessing

In [ ]:
# STAGE 3: EXACT - Define preprocessing function
logger.info("\n" + "="*60)
logger.info("STAGE 3: PREPROCESSING")
logger.info("="*60)

logger.info(f"Preprocessing Configuration:")
logger.info(f"  Target size: {TARGET_SIZE}x{TARGET_SIZE}")
logger.info(f"  Normalization: ImageNet mean={IMAGENET_MEAN}, std={IMAGENET_STD}")

def preprocess_image(image_path, target_size=224):
    """
    Load, resize, and normalize a single image.
    """
    try:
        # Load image
        img = Image.open(image_path)
        
        # Convert RGBA → RGB
        if img.mode == 'RGBA':
            rgb_img = Image.new('RGB', img.size, (255, 255, 255))
            rgb_img.paste(img, mask=img.split()[3])
            img = rgb_img
        elif img.mode == 'L':
            img = img.convert('RGB')
        elif img.mode != 'RGB':
            img = img.convert('RGB')
        
        # Resize
        img_resized = img.resize((target_size, target_size), Image.LANCZOS)
        
        # Convert to numpy array
        img_array = np.array(img_resized, dtype=np.float32)
        
        # Normalize to [0, 1]
        img_array = img_array / 255.0
        
        # ImageNet normalization
        img_array = (img_array - np.array(IMAGENET_MEAN)) / np.array(IMAGENET_STD)
        
        return img_array
        
    except Exception as e:
        logger.error(f"Error processing {image_path}: {e}")
        raise

logger.info("✓ Preprocessing function defined")

In [ ]:
# STAGE 3: EXACT - Preprocess all training images
logger.info(f"\nPreprocessing {len(train_df)} training images...")

X_train = np.zeros((len(train_df), 224, 224, 3), dtype=np.float32)

for idx, row in train_df.iterrows():
    image_path = f"{BASE_INPUT}{row['path']}"
    X_train[idx] = preprocess_image(image_path, target_size=224)
    
    if (idx + 1) % 1000 == 0:
        logger.info(f"  Processed {idx + 1} training images")

logger.info(f"✓ Loaded all training images: X_train.shape = {X_train.shape}")
logger.info(f"  Value range: min={X_train.min():.3f}, max={X_train.max():.3f}")

assert X_train.min() < -1.0, "Preprocessing may have failed (values too high)"
assert X_train.max() > 1.0, "Preprocessing may have failed (values too low)"
assert not np.isnan(X_train).any(), "Found NaN values in X_train!"
assert not np.isinf(X_train).any(), "Found Inf values in X_train!"
logger.info(f"✓ Validation checks passed")

In [ ]:
# STAGE 3: EXACT - Preprocess all test images
logger.info(f"\nPreprocessing {len(test_df)} test images...")

X_test = np.zeros((len(test_df), 224, 224, 3), dtype=np.float32)

for idx, row in test_df.iterrows():
    image_path = f"{BASE_INPUT}{row['path']}"
    X_test[idx] = preprocess_image(image_path, target_size=224)
    
    if (idx + 1) % 500 == 0:
        logger.info(f"  Processed {idx + 1} test images")

logger.info(f"✓ Loaded all test images: X_test.shape = {X_test.shape}")
logger.info(f"  Value range: min={X_test.min():.3f}, max={X_test.max():.3f}")

assert not np.isnan(X_test).any(), "Found NaN values in X_test!"
assert not np.isinf(X_test).any(), "Found Inf values in X_test!"
logger.info(f"✓ Validation checks passed")

In [ ]:
# STAGE 3: EXACT - Save preprocessed data
logger.info("\nSaving preprocessed data...")

os.makedirs('/kaggle/working/preprocessed', exist_ok=True)

np.save('/kaggle/working/preprocessed/X_train_preprocessed.npy', X_train)
np.save('/kaggle/working/preprocessed/X_test_preprocessed.npy', X_test)

logger.info(f"✓ Saved X_train_preprocessed.npy ({X_train.nbytes / 1e9:.2f} GB)")
logger.info(f"✓ Saved X_test_preprocessed.npy ({X_test.nbytes / 1e9:.2f} GB)")

preprocessing_metadata = {
    'target_size': 224,
    'imagenet_mean': IMAGENET_MEAN,
    'imagenet_std': IMAGENET_STD,
    'X_train_shape': list(X_train.shape),
    'X_test_shape': list(X_test.shape),
    'X_train_value_range': [float(X_train.min()), float(X_train.max())],
    'X_test_value_range': [float(X_test.min()), float(X_test.max())],
    'timestamp': str(pd.Timestamp.now())
}

with open('/kaggle/working/preprocessed/normalization_metadata.json', 'w') as f:
    json.dump(preprocessing_metadata, f, indent=2)

logger.info(f"✓ Saved normalization_metadata.json")
logger.info("\n✓ STAGE 3 COMPLETE: Preprocessing finished")

---
# STAGE 4: Train/Validation Stratified Split

In [ ]:
# STAGE 4: EXACT - Create stratified K-fold
logger.info("\n" + "="*60)
logger.info("STAGE 4: TRAIN/VALIDATION STRATIFIED SPLIT")
logger.info("="*60)

kf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

logger.info(f"Creating 5-Fold Stratified Cross-Validation...")
logger.info(f"  n_splits: 5")
logger.info(f"  shuffle: True")
logger.info(f"  random_state: 42")

y_train = train_df['y'].values  # Use numeric class indices

fold_metadata = {}
fold_idx = 0

for train_indices, val_indices in kf.split(X=np.zeros(len(train_df)), y=y_train):
    fold_metadata[f'fold_{fold_idx}'] = {
        'train_indices': train_indices.tolist(),
        'val_indices': val_indices.tolist()
    }
    fold_idx += 1

logger.info(f"Created 5 folds with indices saved")

In [ ]:
# STAGE 4: EXACT - Verify fold balance
logger.info("\nVerifying fold balance...")

for fold_num in range(5):
    fold_key = f'fold_{fold_num}'
    val_indices = np.array(fold_metadata[fold_key]['val_indices'])
    
    val_labels = y_train[val_indices]
    class_counts_fold = np.bincount(val_labels, minlength=10)
    
    assert len(class_counts_fold) == 10, f"Fold {fold_num} missing classes!"
    
    logger.info(f"Fold {fold_num}: ~{class_counts_fold.mean():.0f} images per class")
    
    for class_idx, count in enumerate(class_counts_fold):
        ratio = count / class_counts_fold.mean()
        assert 0.8 < ratio < 1.2, f"Fold {fold_num} class {class_idx} imbalanced ({ratio:.2f}x)"

logger.info("✓ All folds have balanced class distribution")

In [ ]:
# STAGE 4: EXACT - Save fold metadata
logger.info("\nSaving fold metadata...")

with open('/kaggle/working/preprocessed/fold_metadata.json', 'w') as f:
    json.dump(fold_metadata, f, indent=2)

logger.info("✓ Saved fold_metadata.json")
logger.info("\n✓ STAGE 4 COMPLETE: CV splits created")

---
# STAGE 5: Model Training

In [ ]:
# STAGE 5: EXACT - Define augmentation pipelines
logger.info("\n" + "="*60)
logger.info("STAGE 5: MODEL TRAINING")
logger.info("="*60)

train_augmentation = A.Compose([
    A.Rotate(limit=5, p=0.7),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.RandomBrightnessContrast(
        brightness_limit=(-0.2, 0.2),
        contrast_limit=(-0.2, 0.2),
        p=0.5
    ),
    A.GaussianBlur(blur_limit=(3, 3), p=0.3),
    A.RandomCrop(224, 224, p=0.1),
    A.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
        max_pixel_value=1.0
    ),
    ToTensorV2()
], p=1.0)

val_augmentation = A.Compose([
    A.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
        max_pixel_value=1.0
    ),
    ToTensorV2()
], p=1.0)

logger.info("✓ Augmentation pipelines defined")

In [ ]:
# STAGE 5: EXACT - Define Dataset class
class ImageDataset(Dataset):
    def __init__(self, images, labels, augmentation=None):
        self.images = images
        self.labels = labels
        self.augmentation = augmentation
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]
        
        if self.augmentation is not None:
            augmented = self.augmentation(image=image)
            image = augmented['image']
        
        return {
            'image': image,
            'label': torch.tensor(label, dtype=torch.long)
        }

logger.info("✓ ImageDataset class defined")

In [ ]:
# STAGE 5: EXACT - Training loop (ALL 5 FOLDS)
logger.info("\nStarting training loop...")

os.makedirs('/kaggle/working/logs', exist_ok=True)
csv_file = '/kaggle/working/logs/training_log.csv'
csv_header = ['fold', 'epoch', 'train_loss', 'train_acc', 'train_f1', 
              'val_loss', 'val_acc', 'val_f1', 'learning_rate', 'time_sec']

with open(csv_file, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(csv_header)

logger.info(f"Training log will be saved to {csv_file}")

# Loop through folds
for fold_num in range(5):
    logger.info(f"\n{'='*60}")
    logger.info(f"FOLD {fold_num}")
    logger.info(f"{'='*60}")
    
    # Load fold indices
    fold_key = f'fold_{fold_num}'
    train_indices = np.array(fold_metadata[fold_key]['train_indices'])
    val_indices = np.array(fold_metadata[fold_key]['val_indices'])
    
    logger.info(f"Train: {len(train_indices)} | Val: {len(val_indices)}")
    
    # Get train/val labels (already numeric 0-9)
    y_train_fold_idx = train_df.iloc[train_indices]['y'].values
    y_val_fold_idx = train_df.iloc[val_indices]['y'].values
    
    # Get train/val images
    X_train_fold = X_train[train_indices]
    X_val_fold = X_train[val_indices]
    
    # Create datasets
    train_dataset = ImageDataset(X_train_fold, y_train_fold_idx, augmentation=train_augmentation)
    val_dataset = ImageDataset(X_val_fold, y_val_fold_idx, augmentation=val_augmentation)
    
    # Create dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE_TRAIN,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE_VAL,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )
    
    logger.info(f"DataLoaders created: {len(train_loader)} train batches, {len(val_loader)} val batches")
    
    # Initialize model
    model = timm.create_model(MODEL_NAME, pretrained=PRETRAINED, num_classes=NUM_CLASSES, drop_rate=DROPOUT_RATE, drop_path_rate=DROPOUT_PATH_RATE)
    model = model.to(DEVICE)
    
    # Loss function
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    
    # Optimizer
    optimizer = optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        betas=BATA,
        eps=EPS,
        weight_decay=WEIGHT_DECAY
    )
    
    # Scheduler
    scheduler = CosineAnnealingLR(optimizer, T_max=T_MAX, eta_min=1e-6)
    
    logger.info(f"Model initialized: {MODEL_NAME}")
    logger.info(f"Optimizer: AdamW (lr={LEARNING_RATE})")
    logger.info(f"Scheduler: CosineAnnealingLR")
    
    # Create checkpoint directory
    checkpoint_dir = f'/kaggle/working/checkpoints/fold_{fold_num}'
    os.makedirs(checkpoint_dir, exist_ok=True)
    
    # Training loop
    for epoch in range(EPOCHS):
        epoch_start = time.time()
        
        # TRAIN PHASE
        model.train()
        train_loss = 0
        train_preds = []
        train_labels = []
        
        for batch_idx, batch in enumerate(train_loader):
            images = batch['image'].to(DEVICE)
            labels = batch['label'].to(DEVICE)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            train_loss += loss.item()
            train_preds.extend(outputs.argmax(dim=1).detach().cpu().numpy())
            train_labels.extend(labels.detach().cpu().numpy())
        
        train_loss /= len(train_loader)
        train_acc = accuracy_score(train_labels, train_preds)
        train_f1 = f1_score(train_labels, train_preds, average='macro', zero_division=0)
        
        # VAL PHASE
        model.eval()
        val_loss = 0
        val_preds = []
        val_labels = []
        
        with torch.no_grad():
            for batch_idx, batch in enumerate(val_loader):
                images = batch['image'].to(DEVICE)
                labels = batch['label'].to(DEVICE)
                
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                val_preds.extend(outputs.argmax(dim=1).detach().cpu().numpy())
                val_labels.extend(labels.detach().cpu().numpy())
        
        val_loss /= len(val_loader)
        val_acc = accuracy_score(val_labels, val_preds)
        val_f1 = f1_score(val_labels, val_preds, average='macro', zero_division=0)
        
        # Update scheduler
        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']
        
        epoch_time = time.time() - epoch_start
        
        # Log epoch
        logger.info(f"Fold {fold_num} | Epoch {epoch+1:3d}/{EPOCHS} | "
                   f"train_loss: {train_loss:.4f} | val_loss: {val_loss:.4f} | "
                   f"train_acc: {train_acc:.4f} | val_acc: {val_acc:.4f} | "
                   f"train_f1: {train_f1:.4f} | val_f1: {val_f1:.4f} | "
                   f"lr: {current_lr:.2e} | time: {epoch_time:.1f}s")
        
        # Save checkpoint
        checkpoint_path = f'{checkpoint_dir}/epoch_{epoch:03d}.pth'
        torch.save({
            'epoch': epoch,
            'fold': fold_num,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'train_loss': train_loss,
            'train_acc': train_acc,
            'train_f1': train_f1,
            'val_loss': val_loss,
            'val_acc': val_acc,
            'val_f1': val_f1,
            'learning_rate': current_lr,
            'model_name': MODEL_NAME
        }, checkpoint_path)
        
        # Write to CSV
        with open(csv_file, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([fold_num, epoch, train_loss, train_acc, train_f1,
                           val_loss, val_acc, val_f1, current_lr, epoch_time])
    
    logger.info(f"✓ Fold {fold_num} training complete. Saved 100 checkpoints to {checkpoint_dir}/")
    
    # Free memory
    del model, optimizer, scheduler
    del train_dataset, val_dataset, train_loader, val_loader
    torch.cuda.empty_cache()

logger.info(f"\n✓ Training complete for all 5 folds")
logger.info(f"✓ Total checkpoints saved: 500 (5 folds × 100 epochs)")
logger.info(f"✓ Training log: {csv_file}")

---
# STAGE 6: Validation

In [ ]:
# STAGE 6: EXACT - Evaluate all checkpoints
logger.info("\n" + "="*60)
logger.info("STAGE 6: VALIDATION")
logger.info("="*60)

val_augmentation = A.Compose([
    A.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
        max_pixel_value=1.0
    ),
    ToTensorV2()
], p=1.0)

validation_results = []

for fold_num in range(5):
    logger.info(f"\n{'='*60}")
    logger.info(f"VALIDATING FOLD {fold_num}")
    logger.info(f"{'='*60}")
    
    # Load fold indices and data
    fold_key = f'fold_{fold_num}'
    val_indices = np.array(fold_metadata[fold_key]['val_indices'])
    X_val_fold = X_train[val_indices]
    y_val_fold_idx = train_df.iloc[val_indices]['y'].values
    
    # Create dataset and loader
    val_dataset = ImageDataset(X_val_fold, y_val_fold_idx, augmentation=val_augmentation)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE_VAL, shuffle=False, num_workers=2)
    
    checkpoint_dir = f'/kaggle/working/checkpoints/fold_{fold_num}'
    
    # Evaluate all 100 checkpoints for this fold
    for epoch in range(100):
        checkpoint_path = f'{checkpoint_dir}/epoch_{epoch:03d}.pth'
        
        # Load checkpoint
        checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
        
        # Load model
        model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES)
        model.load_state_dict(checkpoint['model_state_dict'])
        model = model.to(DEVICE)
        model.eval()
        
        # Evaluate
        val_preds = []
        val_labels = []
        
        with torch.no_grad():
            for batch in val_loader:
                images = batch['image'].to(DEVICE)
                labels = batch['label'].to(DEVICE)
                
                outputs = model(images)
                val_preds.extend(outputs.argmax(dim=1).detach().cpu().numpy())
                val_labels.extend(labels.detach().cpu().numpy())
        
        # Compute metrics
        accuracy = accuracy_score(val_labels, val_preds)
        f1_macro = f1_score(val_labels, val_preds, average='macro', zero_division=0)
        
        # Get train metrics from checkpoint
        train_acc = checkpoint['train_acc']
        train_f1 = checkpoint['train_f1']
        
        validation_results.append({
            'fold': fold_num,
            'epoch': epoch,
            'train_acc': train_acc,
            'train_f1': train_f1,
            'val_acc': accuracy,
            'val_f1': f1_macro,
            'checkpoint_path': checkpoint_path
        })
        
        if (epoch + 1) % 20 == 0:
            logger.info(f"  Epoch {epoch:3d} | train_acc: {train_acc:.4f} | val_acc: {accuracy:.4f}")
    
    logger.info(f"✓ Validated all 100 checkpoints for fold {fold_num}")
    del model
    torch.cuda.empty_cache()

logger.info(f"\n✓ Validation complete for all 500 checkpoints")

# Convert to DataFrame
val_df = pd.DataFrame(validation_results)
val_df.to_csv('/kaggle/working/logs/validation_metrics.csv', index=False)

logger.info(f"✓ Saved validation_metrics.csv ({len(val_df)} rows)")

---
# STAGE 7: Checkpoint Selection

In [ ]:
# STAGE 7: EXACT - Compute Generalization Score
logger.info("\n" + "="*60)
logger.info("STAGE 7: CHECKPOINT SELECTION")
logger.info("="*60)

val_df['gen_score'] = val_df['val_acc'] - np.abs(val_df['train_acc'] - val_df['val_acc'])

logger.info(f"Generalization Score computed for all checkpoints")
logger.info(f"  Gen_Score range: [{val_df['gen_score'].min():.4f}, {val_df['gen_score'].max():.4f}]")

In [ ]:
# STAGE 7: EXACT - Select best checkpoint per fold
logger.info("\nSelecting best checkpoint per fold...")

best_checkpoints = {}

for fold_num in range(5):
    fold_data = val_df[val_df['fold'] == fold_num].copy()
    
    # Find row with max gen_score
    best_row = fold_data.loc[fold_data['gen_score'].idxmax()]
    
    best_epoch = int(best_row['epoch'])
    best_gen_score = best_row['gen_score']
    best_train_acc = best_row['train_acc']
    best_val_acc = best_row['val_acc']
    best_checkpoint_path = best_row['checkpoint_path']
    
    logger.info(f"Fold {fold_num}: Selected epoch {best_epoch:3d} "
               f"(Gen_Score={best_gen_score:.4f}, train_acc={best_train_acc:.4f}, "
               f"val_acc={best_val_acc:.4f})")
    
    best_checkpoints[fold_num] = {
        'epoch': best_epoch,
        'gen_score': float(best_gen_score),
        'train_acc': float(best_train_acc),
        'val_acc': float(best_val_acc),
        'checkpoint_path': best_checkpoint_path
    }

In [ ]:
# STAGE 7: EXACT - Copy best checkpoints to final_models
logger.info("\nCopying best checkpoints to final_models...")

os.makedirs('/kaggle/working/final_models', exist_ok=True)

for fold_num in range(5):
    best_info = best_checkpoints[fold_num]
    source_path = best_info['checkpoint_path']
    dest_path = f'/kaggle/working/final_models/fold_{fold_num}_best.pth'
    
    shutil.copy(source_path, dest_path)
    logger.info(f"✓ Copied {source_path} → {dest_path}")

In [ ]:
# STAGE 7: EXACT - Save selection rationale
logger.info("\nSaving selection rationale...")

selection_rationale = {
    'formula': 'Gen_Score = val_acc - |train_acc - val_acc|',
    'description': 'Selects checkpoints that generalize well (penalizes overfitting)',
    'selected_folds': {}
}

for fold_num in range(5):
    best_info = best_checkpoints[fold_num]
    selection_rationale['selected_folds'][f'fold_{fold_num}'] = best_info

with open('/kaggle/working/final_models/checkpoint_selection_rationale.json', 'w') as f:
    json.dump(selection_rationale, f, indent=2)

logger.info("✓ Saved checkpoint_selection_rationale.json")

# Also save as CSV
selection_df = pd.DataFrame([
    {
        'fold': f,
        'epoch': best_checkpoints[f]['epoch'],
        'gen_score': best_checkpoints[f]['gen_score'],
        'train_acc': best_checkpoints[f]['train_acc'],
        'val_acc': best_checkpoints[f]['val_acc']
    }
    for f in range(5)
])
selection_df.to_csv('/kaggle/working/final_models/checkpoint_selection_summary.csv', index=False)

logger.info("✓ Saved checkpoint_selection_summary.csv")

In [ ]:
# STAGE 7: EXACT - Verify selected checkpoints
logger.info("\nVerifying selected checkpoints...")

for fold_num in range(5):
    dest_path = f'/kaggle/working/final_models/fold_{fold_num}_best.pth'
    
    # Check file exists
    assert os.path.exists(dest_path), f"Missing {dest_path}"
    
    # Check file size
    file_size_mb = os.path.getsize(dest_path) / 1e6
    assert 90 < file_size_mb < 150, f"Checkpoint size unexpected: {file_size_mb:.0f} MB"
    
    # Load and verify checkpoint structure
    checkpoint = torch.load(dest_path, map_location='cpu')
    assert 'model_state_dict' in checkpoint, "Missing model_state_dict"
    assert 'train_acc' in checkpoint, "Missing train_acc"
    assert 'val_acc' in checkpoint, "Missing val_acc"
    
    logger.info(f"✓ fold_{fold_num}_best.pth verified")

logger.info(f"\n✓ All 5 checkpoints validated successfully")
logger.info(f"\n✓ STAGE 7 COMPLETE: Checkpoint selection finished")

---
# STAGE 8: Inference & Submission

In [ ]:
# STAGE 8: EXACT - Load test data
logger.info("\n" + "="*60)
logger.info("STAGE 8: INFERENCE & SUBMISSION")
logger.info("="*60)

class ImageDatasetTest(Dataset):
    def __init__(self, images, augmentation=None):
        self.images = images
        self.augmentation = augmentation
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        image = self.images[idx]
        
        if self.augmentation is not None:
            augmented = self.augmentation(image=image)
            image = augmented['image']
        
        return image

val_augmentation = A.Compose([
    A.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
        max_pixel_value=1.0
    ),
    ToTensorV2()
], p=1.0)

logger.info("Loading test images...")
test_dataset = ImageDatasetTest(X_test, augmentation=val_augmentation)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE_VAL,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

logger.info(f"✓ Test loader created: {len(test_loader)} batches")

In [ ]:
# STAGE 8: EXACT - Generate predictions from all 5 models
logger.info("\nGenerating predictions from all 5 models...")

test_predictions_all_folds = []

for fold_num in range(5):
    logger.info(f"\nGenerating predictions from fold {fold_num}...")
    
    # Load checkpoint
    checkpoint_path = f'/kaggle/working/final_models/fold_{fold_num}_best.pth'
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    
    # Load model
    model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(DEVICE)
    model.eval()
    
    # Generate predictions
    fold_predictions = []
    
    with torch.no_grad():
        for batch_idx, images in enumerate(test_loader):
            images = images.to(DEVICE)
            
            # Forward pass
            outputs = model(images)
            
            # Get probabilities
            probs = torch.softmax(outputs, dim=1)
            
            # Store probabilities
            fold_predictions.append(probs.detach().cpu().numpy())
            
            if (batch_idx + 1) % 10 == 0:
                logger.info(f"  Batch {batch_idx + 1}/{len(test_loader)}")
    
    # Concatenate all batches for this fold
    fold_probs = np.vstack(fold_predictions)
    test_predictions_all_folds.append(fold_probs)
    
    logger.info(f"✓ Fold {fold_num} predictions: shape={fold_probs.shape}")
    del model
    torch.cuda.empty_cache()

logger.info(f"\n✓ Generated predictions from all 5 folds")

In [ ]:
# STAGE 8: EXACT - Ensemble predictions
logger.info("\nEnsembling predictions...")

# Stack all fold predictions
all_folds_probs = np.stack(test_predictions_all_folds)  # Shape: (5, 3000, 10)

# Average across folds
ensemble_probs = all_folds_probs.mean(axis=0)  # Shape: (3000, 10)

logger.info(f"Ensemble probabilities computed:")
logger.info(f"  Shape: {ensemble_probs.shape}")
logger.info(f"  Mean per sample sums to 1.0? {np.allclose(ensemble_probs.sum(axis=1), 1.0)}")

# Get final predictions
final_predictions = ensemble_probs.argmax(axis=1)  # Shape: (3000,)

logger.info(f"Final predictions computed:")
logger.info(f"  Shape: {final_predictions.shape}")
logger.info(f"  Class distribution: {np.bincount(final_predictions)}")

In [ ]:
# STAGE 8: EXACT - Map predictions to class names
logger.info("\nMapping predictions to class names...")

# Use the predefined class mapping (defined in Stage 1)
# Note: class_idx_to_name already created earlier
final_predictions_names = np.array([class_idx_to_name[idx] for idx in final_predictions])

logger.info(f"Predictions mapped to class names:")
for class_name, count in pd.Series(final_predictions_names).value_counts().items():
    logger.info(f"  {class_name}: {count}")

In [ ]:
# STAGE 8: EXACT - Create submission CSV
logger.info("\nCreating submission CSV...")

os.makedirs('/kaggle/working/submission', exist_ok=True)

# Get test image IDs
test_image_ids = test_df['ID'].values

# Create submission DataFrame
submission_df = pd.DataFrame({
    'ID': test_image_ids,
    'TARGET': final_predictions_names
})

# Save to CSV
submission_path = '/kaggle/working/submission/submission.csv'
submission_df.to_csv(submission_path, index=False)

logger.info(f"\n✓ Submission CSV created: {submission_path}")
logger.info(f"  Shape: {submission_df.shape}")
logger.info(f"  Columns: {list(submission_df.columns)}")
logger.info(f"\n  First 5 rows:")
for idx, row in submission_df.head().iterrows():
    logger.info(f"    {row['ID']}, {row['TARGET']}")

In [ ]:
# STAGE 8: EXACT - Verify submission format
logger.info("\nVerifying submission format...")

# Check 1: Correct number of rows
assert len(submission_df) == 3000, f"Wrong number of rows: {len(submission_df)}"
logger.info(f"✓ 3000 rows")

# Check 2: Correct columns
assert list(submission_df.columns) == ['ID', 'TARGET'], f"Wrong columns: {list(submission_df.columns)}"
logger.info(f"✓ Columns: ID, TARGET")

# Check 3: No missing values
assert not submission_df['ID'].isna().any(), "Missing values in ID column"
assert not submission_df['TARGET'].isna().any(), "Missing values in TARGET column"
logger.info(f"✓ No missing values")

# Check 4: All test IDs present
assert len(submission_df['ID'].unique()) == 3000, "Duplicate IDs in submission"
assert set(submission_df['ID'].values) == set(test_df['ID'].values), "ID mismatch"
logger.info(f"✓ All test IDs present, no duplicates")

# Check 5: All predictions are valid class names
valid_classes = set(class_idx_to_name.values())
invalid_preds = ~submission_df['TARGET'].isin(valid_classes)
assert not invalid_preds.any(), f"Invalid class predictions found"
logger.info(f"✓ All predictions are valid class names")

# Check 6: File exists and is readable
assert os.path.exists(submission_path), f"Submission file not found"
file_size_kb = os.path.getsize(submission_path) / 1e3
logger.info(f"✓ File saved: {file_size_kb:.1f} KB")

logger.info(f"\n{'='*60}")
logger.info(f"✓✓✓ SUBMISSION VERIFIED AND READY FOR KAGGLE ✓✓✓")
logger.info(f"{'='*60}")
logger.info(f"\nNext step: Upload {submission_path} to Kaggle competition")

In [ ]:
# STAGE 8: EXACT - Save metadata and confidence scores
logger.info("\nSaving metadata and confidence scores...")

metadata = {
    'timestamp': str(pd.Timestamp.now()),
    'num_test_samples': len(submission_df),
    'num_classes': 10,
    'ensemble_strategy': '5-fold average',
    'model_architecture': MODEL_NAME,
    'model_source': 'pytorch-image-models (timm)',
    'fold_checkpoints': [f'fold_{f}_best.pth' for f in range(5)]
}

with open('/kaggle/working/submission/submission_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

logger.info(f"✓ Saved submission_metadata.json")

# Save confidence scores
max_probs = ensemble_probs.max(axis=1)
min_probs = ensemble_probs.min(axis=1)
std_probs = ensemble_probs.std(axis=1)

confidence_df = pd.DataFrame({
    'ID': test_image_ids,
    'TARGET': final_predictions_names,
    'max_probability': max_probs,
    'min_probability': min_probs,
    'std_probability': std_probs,
    'confidence_range': max_probs - min_probs
})

confidence_df.to_csv('/kaggle/working/submission/prediction_confidence.csv', index=False)

logger.info(f"✓ Saved prediction_confidence.csv")
logger.info(f"\nConfidence Statistics:")
logger.info(f"  Max prob - mean: {max_probs.mean():.4f}, std: {max_probs.std():.4f}")
logger.info(f"  Confidence range - mean: {(max_probs - min_probs).mean():.4f}")

logger.info(f"\n✓ STAGE 8 COMPLETE: Inference & submission finished")

---
# Pipeline Complete!

In [ ]:
# FINAL: Summary
logger.info("\n" + "="*60)
logger.info("PIPE-1 COMPLETE: END-TO-END PIPELINE FINISHED")
logger.info("="*60)

logger.info(f"""
Pipeline Execution Summary:

✓ Stage 1: Data Loading & Validation
  - Loaded 7,000 training images + 3,000 test images
  - Verified class balance (1,000 per class)
  
✓ Stage 2: EDA
  - Generated visualizations and statistics
  - Output: eda_outputs/ directory
  
✓ Stage 3: Preprocessing
  - Resized all images to 224×224
  - Applied ImageNet normalization
  - Output: preprocessed/ directory (~2.4 GB)
  
✓ Stage 4: Stratified CV Split
  - Created 5-fold stratified cross-validation
  - Verified class balance per fold
  
✓ Stage 5: Model Training
  - Trained EfficientNet-B4 on all 5 folds
  - 100 epochs per fold, 500 checkpoints total
  - Output: checkpoints/ directory (~50 GB)
  
✓ Stage 6: Validation
  - Evaluated all 500 checkpoints
  - Output: logs/validation_metrics.csv
  
✓ Stage 7: Checkpoint Selection
  - Selected best checkpoint per fold using Generalization Score
  - Output: final_models/ directory (5 best checkpoints)
  
✓ Stage 8: Inference & Submission
  - Generated predictions on 3,000 test images
  - Ensemble: averaged probabilities from 5 folds
  - Output: submission/submission.csv (READY FOR KAGGLE!)

Key Files Created:
  - submission/submission.csv ← UPLOAD THIS TO KAGGLE
  - submission/submission_metadata.json
  - submission/prediction_confidence.csv
  - logs/training_log.csv
  - logs/validation_metrics.csv
  - final_models/checkpoint_selection_summary.csv
  - eda_outputs/ (visualizations)

""")

logger.info("Pipeline execution finished at " + str(datetime.now()))
print("\n" + "="*60)
print("✓ PIPE-1 COMPLETE! Ready to submit to Kaggle!")
print("="*60)